# MangaJaNai-Colab
### Upscale ảnh manga trên Google Drive
Chọn MangaJaNai tự động, Eula Digimanga BW V2 NC1 hoặc Manga Ora để Upscale. Chương trình tự tải bộ AI còn thiếu và giữ lại trên Drive.

**Lần đầu sử dụng:**
1. Trong menu **Runtime (Thời gian chạy) → Change runtime type**, chọn **GPU** và **Runtime Version: 2026.07**. Đây là môi trường chương trình đã hỗ trợ.
2. Trên Google Drive, tạo thư mục **Manga/Input** trong **My Drive** rồi tải ảnh vào. Đợi tải lên xong trước khi chạy.
3. Điền các ô ở Bước 1. Bấm nút ▶ ở bên trái từng bước, theo thứ tự **1 → 6**. Chờ bước trước kết thúc rồi mới chạy bước sau.

**Khi thêm ảnh mới vào cùng thư mục:** chạy lại **Bước 5**, rồi **Bước 6** để xem kết quả. Bước 5 tự kết nối lại Drive trước khi tìm ảnh mới; không cần chạy lại Bước 4. Giữ lựa chọn **Chỉ Upscale ảnh chưa làm** ở Bước 1.
Nếu bạn đổi thông tin ở Bước 1, phải bấm ▶ Bước 1 để áp dụng. Nếu Colab bị đặt lại, chạy lại cả 6 bước.

[Hướng dẫn đầy đủ](https://github.com/dex593/MangaJaNai-Colab)


In [ ]:
# @title 1 · Chọn thư mục và cách Upscale
# @markdown ### Ảnh của bạn
# @markdown **Thư mục ảnh gốc:** nơi bạn đã tải ảnh lên Drive. Chương trình đọc cả các thư mục nhỏ bên trong và giữ nguyên ảnh gốc.
# @markdown Nhập đường dẫn như ví dụ dưới đây, **không nhập link chia sẻ Google Drive**. `MyDrive` chính là “My Drive” trong tài khoản của bạn.
Ảnh = "/content/drive/MyDrive/Manga/Input" # @param {type:"string"}
# @markdown **Thư mục lưu ảnh đã Upscale:** ảnh hoàn thành sẽ nằm ở đây. Chương trình tự tạo thư mục nếu chưa có. Chọn thư mục khác với ảnh gốc.
Lưu = "/content/drive/MyDrive/Manga/Output" # @param {type:"string"}
# @markdown **Thư mục lưu AI:** nơi giữ các tệp cần để chương trình Upscale. Chỉ tải những tệp còn thiếu; giữ thư mục này trên Drive để lần sau không phải tải lại.
AI = "/content/drive/MyDrive/MangaJaNai/models" # @param {type:"string"}
# @markdown ### Chọn bộ AI
# @markdown **MangaJaNai:** tự chọn AI theo từng ảnh, dùng được cho thư mục có cả ảnh màu và đen trắng. **Eula Digimanga BW V2 NC1:** chỉ dành cho ảnh đen trắng, hỗ trợ kết quả từ 1 đến 4 lần. **Manga Ora:** dành cho manga, hỗ trợ kết quả từ 1 đến 2 lần.
# @markdown Kết quả Eula và Manga Ora được lưu trong thư mục con riêng tại nơi bạn chọn ở ô **Lưu**, để bạn dễ so sánh và không ghi đè kết quả của bộ AI khác.
Bộ = "MangaJaNai (tự chọn)" # @param ["MangaJaNai (tự chọn)", "Eula Digimanga BW V2 NC1", "2x Manga Ora"]
# @markdown ### Kết quả mong muốn
# @markdown **Upscale gấp mấy lần?** Ví dụ chọn `2`: ảnh 1000 × 1500 sẽ thành 2000 × 3000. Hỗ trợ từ `1` đến `4`, có thể nhập `1.5`.
Upscale = 2 # @param {type:"number"}
# @markdown **Định dạng ảnh:** chọn `png` nếu chưa biết chọn gì. PNG giữ chất lượng tốt; JPEG thường nhẹ hơn nhưng có nén mất chi tiết. AVIF chỉ dùng được khi môi trường hỗ trợ.
Đuôi = "png" # @param ["png", "jpeg", "webp", "avif"]
# @markdown **Cách xử lý ảnh cũ:** thường chọn “Chỉ Upscale ảnh chưa làm” để giữ kết quả đã hoàn tất. Chọn “Upscale lại tất cả ảnh” khi muốn làm lại; kết quả cũ sẽ được thay thế.
Chạy = "Chỉ Upscale ảnh chưa làm" # @param ["Chỉ Upscale ảnh chưa làm", "Upscale lại tất cả ảnh"]
# @markdown ---
# @markdown ### Tùy chọn thêm — có thể giữ nguyên tất cả
# @markdown **Thử lại ảnh bị lỗi:** bật nếu lần trước có ảnh làm chưa thành công và bạn đã xử lý nguyên nhân.
Thử = False # @param {type:"boolean"}
# @markdown **Chỉ xem danh sách:** bật để kiểm tra những ảnh tìm thấy và model sẽ dùng; chưa tải model và chưa tạo ảnh Upscale.
Xem = False # @param {type:"boolean"}
# @markdown **Mức thông báo:** “Thông thường” có tiến độ và kết quả. Chỉ chọn “Tìm lỗi” khi cần gửi thông tin cho người hỗ trợ.
Hiện = "Thông thường" # @param ["Ít thông báo", "Thông thường", "Chi tiết", "Tìm lỗi"]

from pathlib import Path
MODEL, output_folder = {"MangaJaNai (tự chọn)": ("auto", ""), "Eula Digimanga BW V2 NC1": ("eula-digimanga", "Eula Digimanga"), "2x Manga Ora": ("manga-ora", "Manga Ora")}[Bộ]
INPUT_DIR, OUTPUT_DIR, MODEL_CACHE_DIR = Ảnh, str(Path(Lưu) / output_folder), AI
UPSCALE_FACTOR, OUTPUT_FORMAT = Upscale, Đuôi
if MODEL == "manga-ora" and UPSCALE_FACTOR > 2:
    raise ValueError("Manga Ora hỗ trợ từ 1 đến 2 lần. Đổi ô Upscale về 2 hoặc thấp hơn rồi chạy lại Bước 1.")
RESUME, OVERWRITE = {"Chỉ Upscale ảnh chưa làm": (True, False), "Upscale lại tất cả ảnh": (False, True)}[Chạy]
RETRY_FAILED, DRY_RUN = Thử, Xem
STAGING = True
LOG_LEVEL = {"Ít thông báo": "quiet", "Thông thường": "normal", "Chi tiết": "verbose", "Tìm lỗi": "debug"}[Hiện]
TILE_SIZE, GRAYSCALE_THRESHOLD = 0, 12
DETECTOR = "upstream"
PRECISION = "auto"
GITHUB_REPO = "https://github.com/dex593/MangaJaNai-Colab.git"
BRANCH, PIN_COMMIT = "main", ""
PROJECT_DIR = "/content/MangaJaNai-Colab"
print("✓ Đã áp dụng lựa chọn. Tiếp theo: chạy Bước 2 để kết nối Google Drive.")
if OVERWRITE:
    print("Lưu ý: Đã chọn Upscale lại tất cả ảnh. Kết quả cũ sẽ được thay thế.")


In [ ]:
# @title 2 · Kết nối Google Drive
# @markdown Bấm ▶ rồi chọn tài khoản Google có ảnh của bạn. Nếu Google hỏi quyền truy cập, làm theo hướng dẫn trên màn hình.
# @markdown Nếu chương trình báo không tìm thấy ảnh: kiểm tra thư mục trên Drive và đường dẫn ở Bước 1. Để lấy đường dẫn, mở bảng Files bên trái → drive → MyDrive → bấm chuột phải vào thư mục → Copy path. Dán vào Bước 1 rồi chạy lại Bước 1 và Bước 2.
from google.colab import drive
from pathlib import Path
import os
import tempfile

drive.mount("/content/drive")
if not Path(INPUT_DIR).is_dir() or not os.access(INPUT_DIR, os.R_OK):
    raise ValueError(f"Không mở được thư mục ảnh gốc: {INPUT_DIR}. Tạo thư mục trên Drive, tải ảnh vào hoặc sửa đường dẫn ở Bước 1 rồi chạy lại.")
for folder in (OUTPUT_DIR, MODEL_CACHE_DIR):
    Path(folder).mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=folder, prefix=".manga-probe-") as probe:
        probe.write(b"ok")
        probe.flush()
        os.fsync(probe.fileno())
print("✓ Đã kết nối Drive. Đọc được thư mục ảnh gốc và có thể lưu kết quả.")
print("Tiếp theo: chạy Bước 3 để tải chương trình.")


In [ ]:
# @title 3 · Tải bản chương trình mới nhất
# @markdown Bấm ▶ và chờ thông báo hoàn tất. Bước này lấy chương trình từ GitHub; không cần tài khoản GitHub hoặc mật khẩu.
import re
import subprocess
from urllib.parse import urlparse
from pathlib import Path

url = urlparse(GITHUB_REPO)
if url.scheme != "https" or url.netloc != "github.com" or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", url.path):
    raise ValueError("Dùng URL public dạng https://github.com/OWNER/REPO.git")
source_log = Path("/content/mangajanai-logs/source.log")
source_log.parent.mkdir(parents=True, exist_ok=True)
source_log.write_text("", encoding="utf-8")

def git_command(*args):
    result = subprocess.run(["git", *args], env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with source_log.open("a", encoding="utf-8") as log:
        log.write(result.stdout)
    if result.returncode:
        print(result.stdout, flush=True)
        result.check_returncode()
    return result.stdout.strip()

git_command("check-ref-format", "--branch", BRANCH)
if PIN_COMMIT and not re.fullmatch(r"[a-fA-F0-9]{40}", PIN_COMMIT):
    raise ValueError("PIN_COMMIT phải là SHA đầy đủ 40 ký tự, hoặc để rỗng.")
repo = Path(PROJECT_DIR)
print("Đang tải bản chương trình từ GitHub…", flush=True)
if not repo.exists():
    git_command("clone", "--branch", BRANCH, "--", GITHUB_REPO, str(repo))
origin = git_command("-C", str(repo), "remote", "get-url", "origin")
if origin.removesuffix(".git") != GITHUB_REPO.removesuffix(".git"):
    raise ValueError("Thư mục chương trình đang chứa một dự án khác. Hãy mở phiên Colab mới.")
if git_command("-C", str(repo), "status", "--porcelain"):
    raise ValueError("Code chương trình trên Colab đã bị sửa. Lưu phần bạn đã sửa trước khi cập nhật, hoặc mở phiên Colab mới.")
git_command("-C", str(repo), "fetch", "origin", BRANCH)
if PIN_COMMIT:
    git_command("-C", str(repo), "fetch", "origin", PIN_COMMIT)
    git_command("-C", str(repo), "checkout", "--detach", PIN_COMMIT)
else:
    git_command("-C", str(repo), "checkout", BRANCH)
    git_command("-C", str(repo), "pull", "--ff-only", "origin", BRANCH)
git_command("-C", str(repo), "log", "-1", "--format=%h %s")
print("✓ Đã tải chương trình. Tiếp theo: chạy Bước 4.")
print("Log chi tiết:", source_log)


In [ ]:
# @title 4 · Chuẩn bị trước khi Upscale
# @markdown Lần đầu có thể mất vài phút. Bước này cài các thành phần cần thiết và kiểm tra card đồ họa. Chờ thông báo “Sẵn sàng” rồi chạy Bước 5.
from runpy import run_path

run_live = run_path(str(Path(PROJECT_DIR) / "manga_colab/notebook.py"))["run_live"]
print("Đang chuẩn bị. Thông tin kỹ thuật được lưu riêng; nếu có lỗi, hướng dẫn sẽ hiện tại đây.", flush=True)
run_live(["bash", "scripts/setup_colab.sh"], cwd=PROJECT_DIR, check=True, quiet=True,
         log_path="/content/mangajanai-logs/setup.log")
PYTHON = str(Path(PROJECT_DIR) / ".venv-colab/bin/python")
run_live([PYTHON, "-u", "-m", "manga_colab", "doctor", "--input", INPUT_DIR,
          "--output", OUTPUT_DIR, "--models", MODEL_CACHE_DIR], cwd=PROJECT_DIR,
         check=True, log_path="/content/mangajanai-logs/doctor.log")
print("✓ Sẵn sàng. Chạy Bước 5 để Upscale.")


In [ ]:
# @title 5 · Bắt đầu Upscale
# @markdown Bấm ▶ để chạy. Bảng bên dưới sẽ cho biết số ảnh tìm thấy, số ảnh bỏ qua và ảnh đang được Upscale.
# @markdown **Thêm ảnh mới:** đợi tải lên Drive xong rồi chạy lại bước này. Chương trình tự kết nối lại Drive trước khi tìm ảnh; không cần chạy lại Bước 4. Chọn “Chỉ Upscale ảnh chưa làm” ở Bước 1; nếu vừa đổi lựa chọn, nhớ chạy lại Bước 1 trước.
# @markdown Chương trình vẫn đọc ảnh cũ để kiểm tra, nhưng ảnh đã làm xong và còn hợp lệ sẽ không chạy AI lại. Khi muốn dừng, bấm nút ■ và chờ thông báo đã dừng.
from runpy import run_path

launcher = run_path(str(Path(PROJECT_DIR) / "manga_colab/notebook.py"))
run_live = launcher["run_live"]
PYTHON = str(Path(PROJECT_DIR) / ".venv-colab/bin/python")
print(f"Ảnh gốc: {INPUT_DIR}\nLưu kết quả: {OUTPUT_DIR}", flush=True)
print(f"Cách chạy: {Chạy}", flush=True)
print(f"Bộ AI: {Bộ}", flush=True)
args = [PYTHON, "-u", "-m", "manga_colab", "run", "--input", INPUT_DIR,
        "--output", OUTPUT_DIR, "--models", MODEL_CACHE_DIR, "--scale", str(UPSCALE_FACTOR),
        "--format", OUTPUT_FORMAT, "--threshold", str(GRAYSCALE_THRESHOLD),
        "--detector", DETECTOR, "--tile", str(TILE_SIZE), "--precision", PRECISION,
        "--log-level", LOG_LEVEL, "--model", MODEL, "--resume" if RESUME else "--no-resume"]
for enabled, flag in ((OVERWRITE, "--overwrite"), (RETRY_FAILED, "--retry-failed"), (DRY_RUN, "--dry-run")):
    if enabled:
        args.append(flag)
if STAGING:
    args.extend(["--stage", "/content/manga-stage"])
launcher["refresh_drive"](INPUT_DIR)
result = run_live(args, cwd=PROJECT_DIR)
if result.returncode:
    print(f"Lượt Upscale chưa hoàn tất (mã {result.returncode}). Xem nguyên nhân ở thông báo bên trên và Bước 6; ảnh đã hoàn tất được giữ lại.")

In [ ]:
# @title 6 · Xem kết quả
# @markdown Chạy bước này sau Bước 5. Xem ảnh đã làm xong trong “Thư mục lưu ảnh đã Upscale” trên Google Drive. Nếu có ảnh lỗi, bảng sẽ ghi nguyên nhân.
from runpy import run_path

run_live = run_path(str(Path(PROJECT_DIR) / "manga_colab/notebook.py"))["run_live"]
PYTHON = str(Path(PROJECT_DIR) / ".venv-colab/bin/python")
if DRY_RUN:
    print("Dry-run plan:", Path(OUTPUT_DIR) / ".manga-colab/plan.json")
else:
    run_live([PYTHON, "-u", "-m", "manga_colab", "report", "--output", OUTPUT_DIR],
             cwd=PROJECT_DIR, check=True)
